# 04 — Coupling stellar mass loss to the orbits

We bring together the stellar tracks from notebook 01 and the orbits from notebook 03, to simulate a how the orbits evolve with the star's evolution. First we check the mass transfer, then the brown dwarf's perturbations, then the effect of changing the stellar sampling like we did in notebook 01.

Physics in play: slow isotropic mass loss, with quadrupole perturbations in the triple runs. Not yet tides or relativistic precession.




Leave `STELLAR_RUN = None` to find the latest complete, passing 32/64/128 batch for our star, created in notebook 01 (run it till the refinement 128 part). The chosen path is kept fixed for the run. To repeat a particular experiment, set a project relative batch path instead.

The saved AGB suite took me about 13 minutes, the 32–64 pair 13 minutes, and the 64–128 pair 26 minutes. The SeBa replay takes additional time. Each long calculation has its own run cell.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from time import perf_counter
import csv
import inspect
import json
import os
import platform
import subprocess
import sys
import uuid
import warnings
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from amuse.units import units, constants
from amuse.datamodel import Particles
from amuse.community.secularmultiple.interface import SecularMultiple


# None picks the latest complete, matching batch. Use a project-relative folder to pin a run.
PROJECT_ROOT = None  # Normally found from the notebook's working directory.
STELLAR_RUN = None  # Or "data/stellar_sampling_runs/stellar_..."

# Full histories already have NPZ files. Export a combined CSV only if we need it.
EXPORT_FULL_HISTORY_CSV = False

def locate_project_root(project_root=None):
    if project_root is not None:
        candidate = Path(project_root).expanduser().resolve()
        if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
            return candidate
        raise FileNotFoundError(f"PROJECT_ROOT is not a white_dwarf_project folder: {candidate}")
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent, parent / "white_dwarf_project"):
            if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
                return candidate
    raise FileNotFoundError(
        f"Cannot find white_dwarf_project from {cwd}. "
        "Open this notebook from the cloned project, or set PROJECT_ROOT below."
    )


@contextmanager
def working_directory(path):
    previous = Path.cwd()
    Path(path).mkdir(parents=True, exist_ok=True)
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def write_rows(path, rows):
    if not rows:
        raise ValueError("No rows to write.")
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not available in package metadata"


def project_file(root, value):
    """Resolve a saved path inside this copy of the project."""
    root = Path(root).resolve()
    text = str(value).replace("\\", "/")
    if len(text) > 1 and text[1] == ":":
        raise ValueError("Use a project-relative path, not a Windows drive path.")
    path = Path(text).expanduser()
    path = (root / path if not path.is_absolute() else path).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"The selected file must be inside the project: {path}")
    return path


def select_validated_stellar_track(root, track_path=None):
    """Read the passing record for one selected track."""
    root = Path(root).resolve()
    if track_path is None:
        _, path = choose_stellar_batch(root)
    else:
        path = project_file(root, track_path)
    status_path = path.parent.parent / "validation_status.json"
    if not status_path.is_file() or not path.is_file():
        raise FileNotFoundError(
            f"Missing track or validation record for {path}. "
            "Run notebook 01 or copy the complete stellar batch."
        )
    status = json.loads(status_path.read_text(encoding="utf-8"))
    if (status.get("passed_working_tolerances") is not True
        or not status.get("selected_track")
        or project_file(root, status["selected_track"]) != path):
        raise ValueError(f"This track is not selected by a passing notebook-01 report: {path}")
    return path, status_path, status


def choose_stellar_batch(root, stellar_run=None):
    """Find one complete 32/64/128 batch for our 1.2-MSun, Z=0.02 experiment."""
    root = Path(root).resolve()
    sampling_root = root / "data" / "stellar_sampling_runs"

    def check_batch(folder):
        if folder.parent != sampling_root or not folder.is_dir():
            raise FileNotFoundError(f"Choose an existing stellar batch: {folder}")
        status_file = folder / "validation_status.json"
        if not status_file.is_file():
            raise FileNotFoundError(f"Missing notebook-01 validation record: {status_file}")
        status = json.loads(status_file.read_text(encoding="utf-8"))
        if status.get("passed_working_tolerances") is not True or not status.get("selected_track"):
            raise ValueError(f"The stellar summary checks have not passed: {folder.name}")
        selected = project_file(root, status["selected_track"])
        paths = {n: folder / f"refinement_{n:03d}" / "stellar_track.npz" for n in (32, 64, 128)}
        if selected not in paths.values():
            raise ValueError("The selected track must belong to this batch's 32/64/128 sequence.")
        for n, path in paths.items():
            if not path.is_file():
                raise FileNotFoundError(f"Missing refinement {n}: {path}")
            with np.load(path, allow_pickle=False) as data:
                for key, expected in (("initial_mass_MSun", 1.2), ("metallicity", 0.02), ("request_refinement", n)):
                    if (key not in data.files or np.asarray(data[key]).shape != ()
                        or not np.isclose(float(data[key]), expected, rtol=0, atol=1e-12)):
                        raise ValueError(f"Unexpected {key} in {path}")
                kinds = np.asarray(data["stellar_type"])
                if kinds.ndim != 1 or len(kinds) < 2 or kinds[-1] not in (10, 11, 12):
                    raise ValueError(f"The track has not reached a saved WD state: {path}")
        return folder, selected

    if stellar_run is not None:
        text = str(stellar_run).replace("\\", "/")
        if Path(text).is_absolute() or (len(text) > 1 and text[1] == ":"):
            raise ValueError("Use a project-relative STELLAR_RUN folder.")
        return check_batch(project_file(root, text))

    for folder in sorted(sampling_root.glob("stellar_*"), reverse=True):
        try:
            return check_batch(folder.resolve())
        except (OSError, ValueError, KeyError, TypeError, EOFError, zipfile.BadZipFile):
            continue
    raise FileNotFoundError(
        "No complete passing 32/64/128 stellar batch for 1.2 MSun, Z=0.02 was found. "
        "Run notebook 01 through uniform refinement 128, or copy a completed batch "
        "into data/stellar_sampling_runs. Set STELLAR_RUN to its project-relative folder if needed."
    )


def stellar_track_key(value):
    """Keep the project-relative track path from either PC's saved record."""
    text = str(value or "").replace("\\", "/")
    prefix = "data/stellar_sampling_runs/"
    if text.startswith(prefix):
        return text
    if "/" + prefix in text:
        return prefix + text.split("/" + prefix, 1)[1]
    return None


ROOT = locate_project_root(PROJECT_ROOT)
STELLAR_REFINEMENT_DIR, selected_track = choose_stellar_batch(ROOT, STELLAR_RUN)
# Freeze this choice for the replay and every comparison in this kernel.
STELLAR_TRACK = selected_track.relative_to(ROOT).as_posix()
STELLAR_REFINEMENT_RUN = STELLAR_REFINEMENT_DIR.relative_to(ROOT).as_posix()
print("Stellar batch:", STELLAR_REFINEMENT_RUN)
print("Selected track:", STELLAR_TRACK)
WORKER_OPTIONS = {"channel_type": "sockets"}

environment = {
    "python": sys.version, "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": {name: package_version(name) for name in
                 ("amuse-framework", "amuse-secularmultiple", "amuse-seba", "numpy", "matplotlib")},
    "worker_options": WORKER_OPTIONS,
}
for key, arguments in (
    ("git_commit", ["rev-parse", "HEAD"]),
    ("git_branch", ["branch", "--show-current"]),
    ("git_status", ["status", "--porcelain"]),
):
    try:
        environment[key] = subprocess.check_output(
            ["git", *arguments], cwd=ROOT, stderr=subprocess.DEVNULL,
            text=True, timeout=5,
        ).strip()
    except (OSError, subprocess.SubprocessError):
        environment[key] = None
interface_path = Path(inspect.getfile(SecularMultiple)).resolve()
environment["secular_interface_path"] = str(interface_path)

print("Python:", sys.executable)
print("Project root:", ROOT)
print("AMUSE:", environment["packages"]["amuse-framework"])

## 1. What should mass loss do to the orbit?

For the isolated star and planet, let $M=M_\star+m_p$. Isotropic mass loss leaves a central acceleration,

$$\ddot{\mathbf r}=-\frac{GM(t)}{r^3}\mathbf r,$$

so the specific angular momentum $\mathbf h=\mathbf r\times\dot{\mathbf r}$ stays constant. For slow mass loss, the averaged orbit follows

$$\frac{\dot a}{a}=-\frac{\dot M_\star}{M_\star+m_p},\qquad
\langle\dot e\rangle=0,\qquad
a(t)=a_0\frac{M_{\star,0}+m_p}{M_\star(t)+m_p}.$$

We check $aM/(a_0M_0)-1$ and $e-e_0$. For the outer orbit, $M$ also includes the brown dwarf. We also record

$$\Psi=\frac{|\dot M_\star|}{n(M_\star+m_p)},\qquad n=\sqrt{\frac{G(M_\star+m_p)}{a^3}}.$$

A small sampled $\Psi$ cannot rule out rapid changes between saved stellar states. Also, $\mathbf L=\mu\mathbf h$ and the remaining system's energy need not be conserved: mass is leaving the system.

Mostly taken from [Veras et al. (2011)](https://arxiv.org/abs/1107.1239) and [Veras, Hadjidemetriou & Tout (2013)](https://doi.org/10.1093/mnras/stt1451).

In [ ]:
run_id = "mass_loss_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = ROOT / "data" / "mass_loss_validation_runs" / run_id
PLOT_DIR = ROOT / "figures" / "mass_loss_validation_runs" / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
PLOT_DIR.mkdir(parents=True, exist_ok=False)

save_json(RUN_DIR / "environment.json", environment)
print("Output:", RUN_DIR)


## 2. Build the mass-loss control

We prescribe a change from $1.2$ to $0.54\,M_\odot$ over $10^6$ years. The companion masses come from notebook 03; radii stay fixed and positive. Collisions and mutual orbital perturbations are off.

In [ ]:
G = float(constants.G.value_in(units.au**3 / (units.MSun * units.yr**2)))
MP = float((1.9 | units.MEarth).value_in(units.MSun))
MBD = float((27.0 | units.MJupiter).value_in(units.MSun))
RSUN_AU = float((1.0 | units.RSun).value_in(units.au))
BASE = {
    "m_star_MSun": 1.2, "m_planet_MSun": MP, "m_bd_MSun": MBD,
    "a_in_AU": 1.0, "a_out_AU": 20.0,
    "e_in": 1e-3, "e_out": 1e-3,
    "i_in_rad": float(np.deg2rad(65)), "i_out_rad": float(np.deg2rad(5)),
}
M_FINAL = 0.54
LOSS_DURATION_YR = 1e6
HOLD_DURATION_YR = 5e4
RTOL, ATOL_E = 1e-13, 1e-15
COUPLING_INTERVALS = (16, 32, 64)
COMMON_OUTPUT_TIMES = np.linspace(0.0, LOSS_DURATION_YR, 1025)

# Working implementation tolerances, separate from any survival-boundary target.
LIMITS = {"mass_driver_relative": 1e-9, "aM_relative": 1e-7,
          "eccentricity_absolute": 1e-10, "radius_driver_relative": 1e-8}
save_json(RUN_DIR / "experiment_settings.json", {
    "base": BASE, "final_prescribed_mass_MSun": M_FINAL,
    "loss_duration_yr": LOSS_DURATION_YR, "hold_duration_yr": HOLD_DURATION_YR,
    "coupling_intervals": list(COUPLING_INTERVALS),
    "common_output_count": len(COMMON_OUTPUT_TIMES),
    "rtol": RTOL, "atol_e": ATOL_E, "working_tolerances": LIMITS,
    "mutual_orbital_perturbations_enabled": False,
})
print("Analytical final a_in [AU]:", BASE["a_in_AU"]*(BASE["m_star_MSun"]+MP)/(M_FINAL+MP))
print("Analytical final a_out [AU]:", BASE["a_out_AU"]*(BASE["m_star_MSun"]+MP+MBD)/(M_FINAL+MP+MBD))


### Set the initial state and supply the rates

We give the worker `mass_dot_external` and let it evolve the orbit. Applying the analytical expansion as well would count the effect twice.

The interface's `h_vec` includes mass. Its mass-loss term is

$$\left.\frac{\dot{\mathbf L}}{\mathbf L}\right|_{\rm ml}
=\frac{\dot m_1}{m_1}+\frac{\dot m_2}{m_2}
-\frac{\dot m_1+\dot m_2}{m_1+m_2}=\frac{\dot\mu}{\mu}.$$


The short controls use a cumulative clock. The SeBa replay uses a local clock per interval, keeping the total age separately. This avoids losing precision when adding short steps (and also avoids giving me some really weird results due to numerical error!).


In [ ]:
def make_control_hierarchy(cfg, hierarchical=False, star_radius_AU=1.2*RSUN_AU):
    n = 5 if hierarchical else 3
    tree = Particles(n)
    tree.is_binary = [False, False, False, True, True] if hierarchical else [False, False, True]
    tree.child1 = None
    tree.child2 = None
    tree.mass = 0.0 | units.MSun
    tree.radius = 1e-15 | units.au
    for axis in "xyz":
        setattr(tree, f"spin_vec_{axis}", 0.0 | (1/units.yr))
    tree[0].mass = cfg["m_star_MSun"] | units.MSun
    tree[1].mass = cfg["m_planet_MSun"] | units.MSun
    tree[0].radius = star_radius_AU | units.au
    tree[1].radius = 1.2 | units.REarth
    tree.semimajor_axis = 1.0 | units.au
    tree.eccentricity = 1e-3
    tree.inclination = 0.1
    tree.argument_of_pericenter = 0.0
    tree.longitude_of_ascending_node = 0.0
    indices = {"in": 3 if hierarchical else 2}
    if hierarchical:
        tree[2].mass = cfg["m_bd_MSun"] | units.MSun
        tree[2].radius = 1.0 | units.RJupiter
        indices["out"] = 4
    inner = tree[indices["in"]]
    inner.child1, inner.child2 = tree[0], tree[1]
    if hierarchical:
        tree[4].child1, tree[4].child2 = inner, tree[2]
    for suffix, index in indices.items():
        tree[index].semimajor_axis = cfg[f"a_{suffix}_AU"] | units.au
        tree[index].eccentricity = cfg[f"e_{suffix}"]
        tree[index].inclination = cfg[f"i_{suffix}_rad"]
    for field in (
        "include_pairwise_1PN_terms", "include_pairwise_25PN_terms",
        "include_tidal_friction_terms", "include_tidal_bulges_precession_terms",
        "include_rotation_precession_terms", "check_for_secular_breakdown",
        "check_for_dynamical_instability", "check_for_physical_collision_or_orbit_crossing",
        "check_for_minimum_periapse_distance", "check_for_RLOF_at_pericentre",
        "sample_orbital_phases_randomly",
    ):
        setattr(tree, field, False)
    tree.mass_dot_external = 0.0 | (units.MSun/units.yr)
    tree.radius_dot_external = 0.0 | (units.au/units.yr)
    tree.radius_ddot_external = 0.0 | (units.au/units.yr**2)
    return tree, indices


def configure_control(solver, rtol, atol_e):
    solver.parameters.relative_tolerance = rtol
    solver.parameters.absolute_tolerance_eccentricity_vectors = atol_e
    for name in (
        "include_quadrupole_order_terms", "include_octupole_order_binary_pair_terms",
        "include_octupole_order_binary_triplet_terms", "include_hexadecupole_order_binary_pair_terms",
        "include_dotriacontupole_order_binary_pair_terms",
    ):
        setattr(solver.parameters, name, False)


def advance_control(solver, target_yr):
    """Native evolution with explicit checks and no undefined H/H0 diagnostic."""
    if not solver.particles_committed:
        solver.commit_particles()
    target = float(target_yr) | units.yr
    if target < solver.model_time:
        raise ValueError("Cannot evolve backward.")
    end, H, flag, error = solver.evolve_interface(solver.model_time, target-solver.model_time)
    if int(flag) != 0 or int(error) != 0:
        raise RuntimeError(f"SecularMultiple flag={flag}, error_code={error}")
    actual = float(end.value_in(units.yr))
    if not np.isfinite(actual) or not np.isclose(actual, target_yr, rtol=1e-12, atol=1e-7):
        raise RuntimeError(f"Worker returned at {actual}, requested {target_yr} yr.")
    if not np.isfinite(H.value_in(units.MSun*units.au**2/units.yr**2)):
        raise RuntimeError("Non-finite worker Hamiltonian.")
    solver.model_time, solver.hamiltonian = end, H
    solver.flag, solver.error_code = int(flag), int(error)



def advance_control_localclock(solver, target_yr):
    """Advance this control using a local integration clock."""
    if not solver.particles_committed:
        solver.commit_particles()

    target_yr = float(target_yr)
    previous_yr = float(solver.model_time.value_in(units.yr))
    duration_yr = target_yr - previous_yr

    if duration_yr < 0:
        raise ValueError("Cannot evolve backward.")
    if duration_yr == 0:
        return

    # Evolve the existing physical state for this interval.
    elapsed, H, flag, error = solver.evolve_interface(
        0.0 | units.yr,
        duration_yr | units.yr,
    )

    if int(flag) != 0 or int(error) != 0:
        raise RuntimeError(
            f"SecularMultiple flag={flag}, error_code={error}"
        )

    elapsed_yr = float(elapsed.value_in(units.yr))
    if not np.isfinite(elapsed_yr) or not np.isclose(
        elapsed_yr, duration_yr, rtol=1e-12, atol=1e-8
    ):
        raise RuntimeError(
            f"Worker advanced {elapsed_yr} yr; "
            f"requested {duration_yr} yr."
        )

    if not np.isfinite(
        H.value_in(units.MSun * units.au**2 / units.yr**2)
    ):
        raise RuntimeError("Non-finite worker Hamiltonian.")

    # Keep the cumulative age separately from the worker's local clock.
    solver.model_time = target_yr | units.yr
    solver.hamiltonian = H
    solver.flag = int(flag)
    solver.error_code = int(error)


### Pass the sampled history to the worker

We use a constant rate,

$$\dot M_k=\frac{M_{k+1}-M_k}{t_{k+1}-t_k},$$

giving a linear history for mass and radius. Comparing with the supplied history checks the transfer. Comparing with its smooth generating function also checks the sampling approximation.

In [ ]:
def run_mass_loss(label, source_times, source_masses, *, cfg=None,
                  hierarchical=False, source_radii=None, output_times=None,
                  reference_mass=None, rtol=RTOL, atol_e=ATOL_E, provenance=None,
                  advance=advance_control):
    cfg = dict(BASE if cfg is None else cfg)
    times = np.asarray(source_times, dtype=float)
    masses = np.asarray(source_masses, dtype=float)
    radii = np.full_like(masses, 1.2*RSUN_AU) if source_radii is None else np.asarray(source_radii, dtype=float)
    if times.ndim != 1 or len(times) < 2 or masses.shape != times.shape or radii.shape != times.shape:
        raise ValueError("Source times, masses and radii must be equal-length 1D arrays.")
    if not all(np.isfinite(x).all() for x in (times, masses, radii)):
        raise ValueError("Non-finite source values.")
    if times[0] != 0 or not np.all(np.diff(times) > 0) or np.any(masses <= 0) or np.any(radii <= 0):
        raise ValueError("Require t0=0, increasing times and positive masses/radii.")
    if not np.isclose(cfg["m_star_MSun"], masses[0], rtol=1e-13, atol=0):
        raise ValueError("Initial source mass and dynamical mass disagree.")
    requested = times if output_times is None else np.asarray(output_times, dtype=float)
    if requested.ndim != 1 or len(requested) == 0 or not np.isfinite(requested).all() or np.any(np.diff(requested) <= 0):
        raise ValueError("Output times must be finite and increasing.")
    if requested[0] < times[0] or requested[-1] > times[-1]:
        raise ValueError("Output times must lie within the source interval.")
    targets = np.unique(np.concatenate((times, requested)))
    mdot = np.diff(masses)/np.diff(times)
    rdot = np.diff(radii)/np.diff(times)
    folder = RUN_DIR / label
    if folder.exists():
        raise FileExistsError(f"{folder} exists; choose a new label or start a new setup run.")
    folder.mkdir()
    write_rows(folder / "source_nodes.csv", [
        {"time_yr": float(t), "mass_MSun": float(m), "radius_AU": float(r)}
        for t, m, r in zip(times, masses, radii)
    ])
    metadata = {"status": "started", "configuration": cfg, "hierarchical_control": hierarchical,
                "mutual_perturbations": False, "rtol": rtol, "atol_e": atol_e,
                "source_nodes": len(times), "output_count": len(targets),
                "integration_clock": advance.__name__,
                "end_time_yr": float(times[-1]), "source_provenance": provenance,
                "reference_mass": "piecewise-linear driver" if reference_mass is None else "specified smooth law"}
    save_json(folder / "run.json", metadata)
    rows, solver = [], None
    started = perf_counter()
    print(f"{label}: {len(times)-1} coupling intervals, {len(targets)} outputs", flush=True)
    with working_directory(folder):
        try:
            solver = SecularMultiple(**WORKER_OPTIONS)
            configure_control(solver, rtol, atol_e)
            initial, orbit_indices = make_control_hierarchy(cfg, hierarchical, radii[0])
            remote = solver.particles.add_particles(initial)
            initial.new_channel_to(solver.particles).copy()  # Required once, before commitment.
            n_bodies = 3 if hierarchical else 2
            np.testing.assert_allclose(remote[:n_bodies].mass.value_in(units.MSun),
                                       initial[:n_bodies].mass.value_in(units.MSun), rtol=1e-13, atol=0)
            segment = -1
            for target in targets:
                # The rate used for this step belongs to the interval starting at the previous time.
                previous = 0.0 if not rows else rows[-1]["time_yr"]
                new_segment = min(int(np.searchsorted(times, previous, side="right")-1), len(mdot)-1)
                if new_segment != segment:
                    segment = new_segment
                    remote[0].mass_dot_external = float(mdot[segment]) | (units.MSun/units.yr)
                    remote[0].radius_dot_external = float(rdot[segment]) | (units.au/units.yr)
                advance(solver, target)
                mass = float(remote[0].mass.value_in(units.MSun))
                driver = float(np.interp(target, times, masses))
                row = {"time_yr": float(target), "mass_MSun": mass,
                       "driver_mass_MSun": driver,
                       "reference_mass_MSun": driver if reference_mass is None else float(reference_mass(target)),
                       "mass_dot_MSun_yr": float(mdot[segment]),
                       "radius_AU": float(remote[0].radius.value_in(units.au)),
                       "driver_radius_AU": float(np.interp(target, times, radii))}
                if mass <= 0 or row["radius_AU"] <= 0:
                    raise RuntimeError("Worker returned non-positive mass/radius.")
                for suffix, index in orbit_indices.items():
                    orbit = remote[index]
                    a = float(orbit.semimajor_axis.value_in(units.au))
                    e = float(orbit.eccentricity)
                    if a <= 0 or not 0 <= e < 1:
                        raise RuntimeError(f"Invalid bound {suffix} orbit.")
                    extra = cfg["m_planet_MSun"] + (cfg["m_bd_MSun"] if suffix == "out" else 0.0)
                    row[f"a_{suffix}_AU"], row[f"e_{suffix}"] = a, e
                    row[f"a_{suffix}_driver_AU"] = cfg[f"a_{suffix}_AU"]*(masses[0]+extra)/(driver+extra)
                    row[f"a_{suffix}_reference_AU"] = cfg[f"a_{suffix}_AU"]*(masses[0]+extra)/(row["reference_mass_MSun"]+extra)
                    row[f"aM_{suffix}_ratio"] = a*(mass+extra)/(cfg[f"a_{suffix}_AU"]*(masses[0]+extra))
                    n_orbit = np.sqrt(G*(mass+extra)/a**3)
                    row[f"Psi_{suffix}"] = abs(mdot[segment])/(n_orbit*(mass+extra))
                row["q_in_AU"] = row["a_in_AU"]*(1-row["e_in"])
                if not np.isfinite(list(row.values())).all():
                    raise RuntimeError(f"Non-finite diagnostic at {target} yr: {row}")
                rows.append(row)
        except BaseException as exc:
            metadata.update(status="failed", exception=repr(exc), saved_states=len(rows))
            save_json(folder / "run.json", metadata)
            if rows:
                np.savez_compressed(
                    folder / "incomplete_history.npz",
                    **{key: np.array([row[key] for row in rows]) for key in rows[0]},
                )
            raise
        finally:
            if solver is not None:
                try:
                    solver.stop()
                except Exception as exc:
                    warnings.warn(f"Worker cleanup failed: {exc!r}")
    history = {key: np.array([row[key] for row in rows]) for key in rows[0]}
    np.savez_compressed(folder / "history.npz", **history)
    metadata.update(status="completed_control", wall_seconds=perf_counter()-started)
    save_json(folder / "run.json", metadata)
    print(f"  final mass={history['mass_MSun'][-1]:.9f}, a_in={history['a_in_AU'][-1]:.9f} AU", flush=True)
    return history, metadata


def summarize_control(label, history, cfg=BASE):
    h = history
    row = {"run": label, "n_states": len(h["time_yr"]),
           "max_rel_mass_driver_error": float(np.max(abs(h["mass_MSun"]/h["driver_mass_MSun"]-1))),
           "max_rel_mass_reference_error": float(np.max(abs(h["mass_MSun"]/h["reference_mass_MSun"]-1))),
           "max_rel_radius_driver_error": float(np.max(abs(h["radius_AU"]/h["driver_radius_AU"]-1))),
           "final_mass_MSun": float(h["mass_MSun"][-1]), "final_a_in_AU": float(h["a_in_AU"][-1])}
    for suffix in ("in", "out"):
        if f"a_{suffix}_AU" in h:
            row[f"max_aM_{suffix}_error"] = float(np.max(abs(h[f"aM_{suffix}_ratio"]-1)))
            row[f"max_abs_delta_e_{suffix}"] = float(np.max(abs(h[f"e_{suffix}"]-cfg[f"e_{suffix}"])))
            row[f"max_rel_a_{suffix}_reference_error"] = float(np.max(abs(h[f"a_{suffix}_AU"]/h[f"a_{suffix}_reference_AU"]-1)))
            row[f"max_Psi_{suffix}"] = float(h[f"Psi_{suffix}"].max())
    return row


def require_control_accuracy(summary):
    checks = {
        "mass_tracks_driver": summary["max_rel_mass_driver_error"] < LIMITS["mass_driver_relative"],
        "radius_tracks_driver": summary["max_rel_radius_driver_error"] < LIMITS["radius_driver_relative"],
    }
    for suffix in ("in", "out"):
        if f"max_aM_{suffix}_error" in summary:
            checks[f"aM_{suffix}"] = summary[f"max_aM_{suffix}_error"] < LIMITS["aM_relative"]
            checks[f"e_{suffix}"] = summary[f"max_abs_delta_e_{suffix}"] < LIMITS["eccentricity_absolute"]
    if not all(checks.values()):
        raise RuntimeError(f"Control requires investigation: {checks}; {summary}")
    return checks


histories, summaries, checks_by_run = {}, [], {}
def register(label, result, cfg=BASE):
    h, metadata = result
    histories[label] = h
    summary = summarize_control(label, h, cfg)
    save_json(RUN_DIR / label / "summary.json", summary)
    summaries.append(summary)
    checks_by_run[label] = require_control_accuracy(summary)
    return h


## 3. Check the histories

### Fixed mass, linear loss and the outer orbit

A linear history is represented exactly by its interval rates. The hierarchy test checks both orbits with their mutual perturbations still off.

Should take about 1-2 minutes.

In [ ]:
fixed = register("01_fixed_mass", run_mass_loss(
    "01_fixed_mass", [0, LOSS_DURATION_YR], [BASE["m_star_MSun"]]*2,
    output_times=np.linspace(0, LOSS_DURATION_YR, 129),
))
linear_times = np.array([0.0, LOSS_DURATION_YR, LOSS_DURATION_YR+HOLD_DURATION_YR])
linear_masses = np.array([BASE["m_star_MSun"], M_FINAL, M_FINAL])
linear_outputs = np.unique(np.concatenate((
    np.linspace(0, LOSS_DURATION_YR, 257),
    np.linspace(LOSS_DURATION_YR, linear_times[-1], 17),
)))
linear = register("02_linear_binary", run_mass_loss(
    "02_linear_binary", linear_times, linear_masses, output_times=linear_outputs,
))
hierarchy = register("03_linear_hierarchy", run_mass_loss(
    "03_linear_hierarchy", linear_times, linear_masses,
    output_times=linear_outputs, hierarchical=True,
))
print(json.dumps(summaries, indent=2))

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
t = linear["time_yr"]/1e6
axes[0, 0].plot(t, linear["mass_MSun"])
axes[0, 0].set_ylabel(r"Stellar mass [$M_\odot$]")
axes[0, 1].plot(t, linear["a_in_AU"], label="Worker")
axes[0, 1].plot(t, linear["a_in_driver_AU"], "--", label="Analytical")
axes[0, 1].set_ylabel("Inner semimajor axis [AU]")
axes[0, 1].legend(frameon=False)
for suffix in ("in", "out"):
    axes[1, 0].plot(hierarchy["time_yr"]/1e6, hierarchy[f"aM_{suffix}_ratio"]-1, label=suffix)
axes[1, 0].set_ylabel(r"$aM/(a_0 M_0)-1$")
axes[1, 0].legend(frameon=False)
axes[1, 1].plot(t, linear["e_in"]-BASE["e_in"])
axes[1, 1].set_ylabel(r"$e-e_0$")
for ax in axes[1]: ax.set_xlabel("Time [Myr]")
fig.tight_layout()
fig.savefig(PLOT_DIR / "linear_mass_loss_controls.png", dpi=180)
plt.show()


### Refine a curved history

We use

$$M_\star(t)=M_0e^{-\kappa t},\qquad \kappa=\frac{\ln(M_0/M_f)}{T}.$$

We double the source intervals from 16 to 32 to 64 while keeping 1,025 output times. Linear interpolation of this smooth function should improve roughly as $\Delta t^2$. We then tighten the solver tolerances.

In [ ]:
KAPPA = np.log(BASE["m_star_MSun"]/M_FINAL)/LOSS_DURATION_YR
def exponential_mass(t):
    return BASE["m_star_MSun"]*np.exp(-KAPPA*np.asarray(t))

refinement_rows = []
for n_intervals in COUPLING_INTERVALS:
    source_t = np.linspace(0, LOSS_DURATION_YR, n_intervals+1)
    label = f"04_exponential_n{n_intervals:03d}"
    h = register(label, run_mass_loss(
        label, source_t, exponential_mass(source_t),
        output_times=COMMON_OUTPUT_TIMES, reference_mass=exponential_mass,
    ))
    summary = summarize_control(label, h)
    refinement_rows.append({"n_coupling_intervals": n_intervals,
        "coupling_step_yr": LOSS_DURATION_YR/n_intervals,
        "max_rel_mass_reference_error": summary["max_rel_mass_reference_error"],
        "max_rel_a_reference_error": summary["max_rel_a_in_reference_error"],
        "max_aM_error": summary["max_aM_in_error"]})

write_rows(RUN_DIR / "coupling_refinement.csv", refinement_rows)
print(json.dumps(refinement_rows, indent=2))
coupling_orders = []
for coarse, fine in zip(refinement_rows[:-1], refinement_rows[1:]):
    p = np.log(coarse["max_rel_mass_reference_error"]/fine["max_rel_mass_reference_error"])/np.log(
        coarse["coupling_step_yr"]/fine["coupling_step_yr"])
    coupling_orders.append(float(p))
print("Measured mass-interpolation orders:", coupling_orders)
if not all(b["max_rel_mass_reference_error"] < a["max_rel_mass_reference_error"]
           for a, b in zip(refinement_rows[:-1], refinement_rows[1:])):
    raise RuntimeError("Coupling refinement did not reduce mass-history error.")

source_t = np.linspace(0, LOSS_DURATION_YR, COUPLING_INTERVALS[-1]+1)
tight = register("05_exponential_tight", run_mass_loss(
    "05_exponential_tight", source_t, exponential_mass(source_t),
    output_times=COMMON_OUTPUT_TIMES, reference_mass=exponential_mass,
    rtol=RTOL/10, atol_e=ATOL_E/10,
))
standard = histories[f"04_exponential_n{COUPLING_INTERVALS[-1]:03d}"]
np.testing.assert_allclose(standard["time_yr"], tight["time_yr"], rtol=0, atol=1e-7)
tolerance_comparison = {
    "max_rel_a_difference": float(np.max(abs(standard["a_in_AU"]/tight["a_in_AU"]-1))),
    "max_rel_mass_difference": float(np.max(abs(standard["mass_MSun"]/tight["mass_MSun"]-1))),
}
save_json(RUN_DIR / "tolerance_comparison.json", tolerance_comparison)
print("Fixed-driver tolerance comparison:", tolerance_comparison)
if tolerance_comparison["max_rel_a_difference"] >= LIMITS["aM_relative"]:
    raise RuntimeError("Solver-tolerance comparison exceeds the working orbital tolerance.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for n in COUPLING_INTERVALS:
    h = histories[f"04_exponential_n{n:03d}"]
    axes[0].plot(h["time_yr"]/1e6, h["a_in_AU"]/h["a_in_reference_AU"]-1, label=f"{n} intervals")
axes[0].set(xlabel="Time [Myr]", ylabel="Relative error against smooth-law a(t)")
axes[0].legend(frameon=False)
steps = np.array([r["coupling_step_yr"] for r in refinement_rows])
errors = np.array([r["max_rel_mass_reference_error"] for r in refinement_rows])
axes[1].loglog(steps, errors, "o-", label="Measured")
axes[1].loglog(steps, errors[-1]*(steps/steps[-1])**2, "--", label=r"$\Delta t^2$ reference")
axes[1].set(xlabel="Coupling interval [yr]", ylabel="Maximum relative mass-history error")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "coupling_refinement.png", dpi=180)
plt.show()


### Change the output cadence

Keep the 64 source intervals and reduce the outputs from 1,025 to 513. We compare shared times.

In [ ]:
sparse = register("06_exponential_sparse_output", run_mass_loss(
    "06_exponential_sparse_output", source_t, exponential_mass(source_t),
    output_times=COMMON_OUTPUT_TIMES[::2], reference_mass=exponential_mass,
))
np.testing.assert_allclose(sparse["time_yr"], standard["time_yr"][::2], rtol=0, atol=1e-7)
output_comparison = {
    "max_rel_a_difference": float(np.max(abs(sparse["a_in_AU"]/standard["a_in_AU"][::2]-1))),
    "max_rel_mass_difference": float(np.max(abs(sparse["mass_MSun"]/standard["mass_MSun"][::2]-1))),
}
save_json(RUN_DIR / "output_comparison.json", output_comparison)
print(output_comparison)
if output_comparison["max_rel_a_difference"] >= LIMITS["aM_relative"]:
    raise RuntimeError("Output-call comparison exceeds the working orbital tolerance.")


## 4. Replay the saved SeBa track

Now we supply the selected mass and radius history from notebook 01, keeping every source knot. This checks the one-way transfer into SecularMultiple; adding orbital substeps cannot recover missing stellar evolution.

Should take oughly 10–20 minutes .

In [ ]:
# Use the same input settings on every machine.
SEBA_TRACK_PATH, SEBA_VALIDATION_PATH, sampling_status = select_validated_stellar_track(
    ROOT, STELLAR_TRACK,
)

print("Using:", SEBA_TRACK_PATH.relative_to(ROOT))

path = Path(SEBA_TRACK_PATH).expanduser().resolve()
with np.load(path, allow_pickle=False) as data:
    required = ("time_Myr", "mass_MSun", "radius_RSun")
    missing = [key for key in required if key not in data.files]
    if missing:
        raise KeyError(f"Missing source columns: {missing}")
    age = np.array(data["time_Myr"], dtype=float)
    mass = np.array(data["mass_MSun"], dtype=float)
    radius = np.array(data["radius_RSun"], dtype=float)*RSUN_AU
    kinds = np.array(data["stellar_type"]) if "stellar_type" in data.files else None
if age.ndim != 1 or len(age) < 2 or mass.shape != age.shape or radius.shape != age.shape:
    raise ValueError("Invalid SeBa source array shapes.")
if kinds is not None and (kinds.shape != age.shape or not np.isfinite(kinds).all()):
    raise ValueError("Invalid SeBa stellar-type array.")
print("Source states:", len(age))
print("Final stellar mass [MSun]:", mass[-1])
local_time = (age-age[0])*1e6
replay_cfg = {**BASE, "m_star_MSun": float(mass[0])}
provenance = {"kind": "saved_SeBa_track", "path": path.relative_to(ROOT).as_posix(),
              "stellar_age_offset_Myr": float(age[0]),
              "stellar_track_convergence": sampling_status["status"],
                "stellar_validation_path": SEBA_VALIDATION_PATH.relative_to(ROOT).as_posix()}
replay_started = perf_counter()
replay = register("07_seba_track_replay_localclock", run_mass_loss(
    "07_seba_track_replay_localclock", local_time, mass, cfg=replay_cfg,
    source_radii=radius, provenance=provenance,
    advance=advance_control_localclock,
), cfg=replay_cfg)
print(f"Replay time: {(perf_counter()-replay_started)/60:.1f} min")
seba_replay_status = {
    "status": "completed_transfer_control", "provenance": provenance,
    "source_intervals": len(age)-1,
    "max_adjacent_mass_fraction": float(np.max(abs(np.diff(mass))/mass[:-1])),
    "phase_changes": None if kinds is None else int(np.count_nonzero(np.diff(kinds))),
    "max_sampled_interval_Psi": float(replay["Psi_in"].max()),
    "note": "Small interval-average Psi cannot exclude unresolved rapid mass loss.",
}
print(json.dumps(seba_replay_status, indent=2))
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(age/1000, replay["a_in_AU"], label="SecularMultiple")
axes[0].plot(age/1000, replay["a_in_driver_AU"], "--", label="Analytical from saved masses")
axes[0].set_ylabel("Semimajor axis [AU]")
axes[0].legend(frameon=False)
axes[1].plot(age/1000, replay["aM_in_ratio"]-1)
axes[1].set(xlabel="Stellar age [Gyr]", ylabel=r"$aM/(a_0 M_0)-1$")
fig.tight_layout()
fig.savefig(PLOT_DIR / "seba_replay_control.png", dpi=180)
plt.show()
save_json(RUN_DIR / "seba_replay_status.json", seba_replay_status)


## 5. Save the transfer results

In [ ]:
# Reload the completed replay from this run's saved files.
label = "07_seba_track_replay_localclock"
folder = RUN_DIR / label
run_info = json.loads((folder / "run.json").read_text())

with np.load(folder / "history.npz", allow_pickle=False) as data:
    current_history = {key: data[key].copy() for key in data.files}

current_summary = summarize_control(
    label, current_history, cfg=run_info["configuration"]
)
current_checks = require_control_accuracy(current_summary)

# Replace any earlier summary for the same replay.
histories[label] = current_history
summaries = [row for row in summaries if row["run"] != label]
summaries.append(current_summary)
checks_by_run[label] = current_checks

seba_replay_status = json.loads(
    (RUN_DIR / "seba_replay_status.json").read_text()
)
assert (
    stellar_track_key(seba_replay_status["provenance"]["path"]) is not None
    and stellar_track_key(seba_replay_status["provenance"]["path"])
    == stellar_track_key(run_info["source_provenance"]["path"])
), "Replay status and saved history refer to different source tracks."


export_dir = RUN_DIR / "readable_export"
export_dir.mkdir(exist_ok=True)
columns = list(dict.fromkeys(key for row in summaries for key in row))
write_rows(export_dir / "summary.csv", [
    {key: row.get(key) for key in columns} for row in summaries
])
# This optional table is the only full-history CSV export.
if EXPORT_FULL_HISTORY_CSV:
    history_columns = ["run"] + sorted({key for h in histories.values() for key in h})
    with (export_dir / "histories.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=history_columns)
        writer.writeheader()
        for label, h in histories.items():
            for i in range(len(h["time_yr"])):
                writer.writerow({"run": label, **{key: values[i].item() for key, values in h.items()}})

report = {
    "status": "completed_control_suite",
    "scope": "adiabatic mass-loss transfer controls; no survival inference",
    "checks_by_run": checks_by_run, "coupling_orders": coupling_orders,
    "solver_tolerance_comparison": tolerance_comparison,
    "output_call_comparison": output_comparison,
    "seba_replay": seba_replay_status,
}
save_json(RUN_DIR / "validation_status.json", report)
run_records = {
    label: json.loads((RUN_DIR / label / "run.json").read_text())
    for label in histories
}
save_json(export_dir / "metadata.json", {
    "environment": environment, "validation": report,
    "settings": json.loads((RUN_DIR / "experiment_settings.json").read_text()),
    "runs": run_records,
    "file_reference_base": "metadata.json directory (source_nodes, history_files, full_history_csv, incomplete_histories)",
    "source_nodes": {
        label: {"path": f"../{label}/source_nodes.csv", "n_rows": info["source_nodes"]}
        for label, info in run_records.items()
    },
    "history_files": {
        label: {"path": f"../{label}/history.npz", "n_states": len(h["time_yr"])}
        for label, h in histories.items()
    },
    "full_history_csv": "histories.csv" if EXPORT_FULL_HISTORY_CSV else None,
    "units": {"time_yr": "yr", "mass_MSun": "MSun", "mass_dot_MSun_yr": "MSun/yr",
              "columns_ending_AU": "AU", "aM_ratios_and_Psi": "dimensionless"},
    "incomplete_histories": [
        "../" + p.relative_to(RUN_DIR).as_posix()
        for p in sorted(RUN_DIR.glob("*/incomplete_history.*"))
        if p.suffix in (".npz", ".csv")
    ],
})
print(f"Saved reports for {len(histories)} completed controls.")
exported_names = ["summary.csv", "metadata.json"]
if EXPORT_FULL_HISTORY_CSV:
    exported_names.append("histories.csv")
else:
    print("Full CSV export skipped; numerical histories are saved in each case's history.npz.")
for name in exported_names:
    print(export_dir / name)


# Display the CSV that was actually written.
print("\nSaved summary:", export_dir / "summary.csv")
print((export_dir / "summary.csv").read_text())


## 6. Add the brown dwarf during the AGB

The window runs from the first saved AGB state to the first WD state, hopefully. We start with $a_{\rm in}=5$ au, $a_{\rm out}=100$ au, both eccentricities $10^{-3}$, and mutual inclination $60^\circ$. The companions are $1.9\,M_\oplus$ and $27\,M_{\rm Jup}$.

**These orbits start at the AGB window**. We follow

$$q=a_{\rm in}(1-e_{\rm in}),\qquad C=q-R_\star-R_p.$$

$C=0$ means geometric surface contact, not necesarilly engulfment. We also stop for orbit crossing, instability or secular breakdown which means the math has some error, I think



Preparation requires a passing stellar summary record and a completed transfer check for the same source path. It keeps every AGB source knot and creates a new output folder.

In [ ]:
COUPLED_IMPLEMENTATION = "notebook04-agb-quadrupole-v1"

COUPLED_G = 39.4852492465  # AU^3 / (MSun yr^2), as in the AMUSE SecularMultiple example

COUPLED_RSUN_AU = 695500000.0 / 149597870691.0

COUPLED_RPLANET_AU = 1.2 * 6371000.0 / 149597870691.0

COUPLED_RBD_AU = 69911000.0 / 149597870691.0

COUPLED_DEFAULTS = {
    "m_planet_MSun": 5.7051967902178065e-6,
    "m_bd_MSun": 0.025775244856505036,
    "a_in_AU": 5.0, "a_out_AU": 100.0,
    "e_in": 1e-3, "e_out": 1e-3,
    "mutual_inclination_deg": 60.0, "outer_inclination_deg": 5.0,
}

COUPLED_LIMITS = {
    "mass_driver_relative": 1e-9, "radius_driver_relative": 1e-8,
    "aM_relative": 1e-7, "fixed_H_relative": 1e-7,
    "fixed_J_relative": 1e-7, "benchmark_emax_absolute": 0.005,
    "comparison_e_absolute": 1e-3, "comparison_q_AU": 1e-3,
    "comparison_a_relative": 1e-7, "event_time_yr": 10.0,
}


def coupled_save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def coupled_save_rows(path, rows):
    columns = list(dict.fromkeys(k for row in rows for k in row))
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


@contextmanager
def coupled_working_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def coupled_lk_time(mass, a_in, a_out, cfg):
    m_inner = mass + cfg["m_planet_MSun"]
    m_total = m_inner + cfg["m_bd_MSun"]
    p_in = 2*np.pi*np.sqrt(a_in**3/(COUPLED_G*m_inner))
    p_out = 2*np.pi*np.sqrt(a_out**3/(COUPLED_G*m_total))
    return (8/(15*np.pi))*(m_total/cfg["m_bd_MSun"])*p_out**2/p_in*(1-cfg["e_out"]**2)**1.5


In [ ]:
def coupled_load_agb_window(path):
    with np.load(path, allow_pickle=False) as data:
        age, mass, radius, kinds = [np.asarray(data[k], dtype=float).copy() for k in
                                   ("time_Myr", "mass_MSun", "radius_RSun", "stellar_type")]
    if age.ndim != 1 or len(age) < 2 or any(x.shape != age.shape for x in (mass, radius, kinds)):
        raise ValueError("Invalid stellar track shapes.")
    if not all(np.isfinite(x).all() for x in (age, mass, radius, kinds)):
        raise ValueError("Non-finite stellar track.")
    if np.any(np.diff(age) <= 0) or np.any(mass <= 0) or np.any(radius <= 0):
        raise ValueError("Require increasing ages and positive masses/radii.")
    agb = np.flatnonzero(np.isin(kinds, (5, 6)))
    if not len(agb):
        raise ValueError("No AGB samples (stellar types 5 or 6).")
    start = int(agb[0])
    wd = np.flatnonzero(np.isin(kinds, (10, 11, 12)) & (np.arange(len(age)) > start))
    if not len(wd):
        raise ValueError("No white-dwarf sample after the AGB window.")
    end = int(wd[0])
    sl = slice(start, end+1)
    return {"time_yr": (age[sl]-age[start])*1e6, "mass_MSun": mass[sl],
            "radius_AU": radius[sl]*COUPLED_RSUN_AU, "stellar_type": kinds[sl],
            "age_offset_Myr": float(age[start]), "source_start_index": start,
            "source_end_index": end}


def coupled_prepare(root, track_path=None,
                    a_in_AU=5.0, a_out_AU=100.0, inclination_deg=60.0):
    """Prepare a separate output folder, without running the integrator."""
    root = Path(root).resolve()
    path, validation_path, stellar_status = select_validated_stellar_track(root, track_path)
    # Match a completed transfer control by the saved track path.
    transfer_path = None
    for p in sorted((root/"data/mass_loss_validation_runs").glob("mass_loss_*/validation_status.json"), reverse=True):
        report = json.loads(p.read_text())
        replay = report.get("seba_replay", {})
        if stellar_track_key(replay.get("provenance", {}).get("path")) != path.relative_to(root).as_posix():
            continue
        checks = report.get("checks_by_run", {}).get("07_seba_track_replay_localclock", {})
        if (report.get("status") == "completed_control_suite"
                and replay.get("status") == "completed_transfer_control"
                and all(checks.get(k) is True for k in ("mass_tracks_driver", "radius_tracks_driver", "aM_in", "e_in"))):
            transfer_path = p
            break
    if transfer_path is None:
        raise RuntimeError("No passing notebook-04 replay found for this track run. Run sections 1–5, or copy their completed run folder to this machine.")
    window = coupled_load_agb_window(path)
    cfg = {**COUPLED_DEFAULTS, "a_in_AU": float(a_in_AU), "a_out_AU": float(a_out_AU),
           "mutual_inclination_deg": float(inclination_deg), "m_star_MSun": float(window["mass_MSun"][0])}
    if not 0 < cfg["a_in_AU"] < cfg["a_out_AU"] or not 0 <= inclination_deg <= 175:
        raise ValueError("Invalid orbital initial conditions.")
    if cfg["a_in_AU"]*(1-cfg["e_in"]) <= window["radius_AU"][0]+COUPLED_RPLANET_AU:
        raise ValueError("The proposed orbit is already in surface contact at the window start.")
    run_id = "coupled_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
    folder = root/"data/coupled_mass_loss_runs"/run_id
    folder.mkdir(parents=True, exist_ok=False)
    provenance = {"source_track": path.relative_to(root).as_posix(),
                  "stellar_validation": validation_path.relative_to(root).as_posix(),
                  "transfer_validation": transfer_path.relative_to(root).as_posix()}
    plan = {"root": root, "folder": folder, "cfg": cfg, "window": window, "provenance": provenance}
    settings = {"configuration_at_window_start": cfg, "source": provenance,
                "age_offset_Myr": window["age_offset_Myr"], "window_end_age_Myr": float(window["age_offset_Myr"]+window["time_yr"][-1]/1e6),
                "source_start_index": window["source_start_index"], "source_end_index": window["source_end_index"],
                "source_states_in_window": len(window["time_yr"]), "working_tolerances": COUPLED_LIMITS,
                "order": "quadrupole", "tides": False, "relativity": False,
                "orbital_initial_conditions": "specified at first saved AGB sample; not propagated from the main sequence",
                "scope": "numerical convergence for a fixed piecewise-linear stellar driver; no survival claim",
                "integration_clock": "zero origin for each interval; cumulative age kept separately"}
    coupled_save_json(folder/"experiment_settings.json", settings)
    coupled_save_json(folder/"stellar_validation.json", stellar_status)
    coupled_save_json(folder/"transfer_validation.json", json.loads(transfer_path.read_text()))
    np.savez_compressed(folder/"source_window.npz", **window)
    print("Track:", provenance["source_track"])
    print(f"AGB window: {window['age_offset_Myr']:.6f} to {settings['window_end_age_Myr']:.6f} Myr")
    print(f"Source states: {len(window['time_yr']):,}; initial a_in={a_in_AU:g}, a_out={a_out_AU:g} AU")
    print("These orbital elements apply at the start of this window.")
    print("Results:", folder)
    return plan


### Build the coupled worker

Set the physical state once, then supply mass and radius rates using a local clock.

In [ ]:
class CoupledAmuseBackend:
    """Keep AMUSE calls separate from the interval and comparison logic."""
    def __init__(self, cfg, mass, radius, quadrupole, rtol, atol, events, worker_options):
        from amuse.units import units
        from amuse.datamodel import Particles
        from amuse.community.secularmultiple.interface import SecularMultiple
        self.units = units
        self.solver = SecularMultiple(**worker_options)
        self.H = 0.0
        try:
            p = self.solver.parameters
            p.relative_tolerance = rtol
            p.absolute_tolerance_eccentricity_vectors = atol
            p.include_quadrupole_order_terms = quadrupole
            for name in ("include_octupole_order_binary_pair_terms", "include_octupole_order_binary_triplet_terms",
                         "include_hexadecupole_order_binary_pair_terms", "include_dotriacontupole_order_binary_pair_terms"):
                setattr(p, name, False)
            tree = Particles(5)
            tree.is_binary = [False, False, False, True, True]
            tree.child1 = None
            tree.child2 = None
            tree.mass = [mass, cfg["m_planet_MSun"], cfg["m_bd_MSun"], 0, 0] | units.MSun
            tree.radius = [radius, COUPLED_RPLANET_AU, COUPLED_RBD_AU, 1e-15, 1e-15] | units.au
            for axis in "xyz":
                setattr(tree, "spin_vec_"+axis, 0.0 | (1/units.yr))
            tree.semimajor_axis = 1.0 | units.au
            tree.eccentricity = 1e-3
            tree.inclination = 0.1
            tree.argument_of_pericenter = 0.0
            tree.longitude_of_ascending_node = 0.0
            tree[3].child1, tree[3].child2 = tree[0], tree[1]
            tree[4].child1, tree[4].child2 = tree[3], tree[2]
            for suffix, i, inc in (("in", 3, cfg["outer_inclination_deg"]+cfg["mutual_inclination_deg"]),
                                   ("out", 4, cfg["outer_inclination_deg"])):
                tree[i].semimajor_axis = cfg["a_"+suffix+"_AU"] | units.au
                tree[i].eccentricity = cfg["e_"+suffix]
                tree[i].inclination = float(np.deg2rad(inc))
            for name in ("include_pairwise_1PN_terms", "include_pairwise_25PN_terms", "include_tidal_friction_terms",
                         "include_tidal_bulges_precession_terms", "include_rotation_precession_terms",
                         "check_for_secular_breakdown", "check_for_dynamical_instability",
                         "check_for_physical_collision_or_orbit_crossing", "check_for_minimum_periapse_distance",
                         "check_for_RLOF_at_pericentre", "sample_orbital_phases_randomly"):
                setattr(tree, name, False)
            tree.mass_dot_external = 0.0 | (units.MSun/units.yr)
            tree.radius_dot_external = 0.0 | (units.au/units.yr)
            tree.radius_ddot_external = 0.0 | (units.au/units.yr**2)
            if events:
                tree[3].check_for_physical_collision_or_orbit_crossing = True
                tree[4].check_for_physical_collision_or_orbit_crossing = True
                tree[3].check_for_secular_breakdown = quadrupole
                tree[3].check_for_dynamical_instability = True
                tree[3].dynamical_instability_criterion = 0
            self.remote = self.solver.particles.add_particles(tree)
            tree.new_channel_to(self.solver.particles).copy()
            self.solver.commit_particles()
            self.H = float(self.solver.initial_hamiltonian.value_in(units.MSun*units.au**2/units.yr**2))
        except BaseException:
            self.stop()
            raise

    def set_rates(self, mdot, rdot):
        self.remote[0].mass_dot_external = float(mdot) | (self.units.MSun/self.units.yr)
        self.remote[0].radius_dot_external = float(rdot) | (self.units.au/self.units.yr)

    def advance(self, dt):
        u = self.units
        elapsed, H, flag, error = self.solver.evolve_interface(0.0 | u.yr, float(dt) | u.yr)
        elapsed = float(elapsed.value_in(u.yr))
        self.H = float(H.value_in(u.MSun*u.au**2/u.yr**2))
        flag, error = int(flag), int(error)
        if error != 0 or flag not in (0, 2):
            raise RuntimeError(f"SecularMultiple flag={flag}, error={error}")
        events = []
        if flag == 2:
            for i, name in ((3, "surface_contact"), (4, "orbit_crossing")):
                if self.remote[i].physical_collision_or_orbit_crossing_has_occurred:
                    events.append(name)
            for attribute, name in (("secular_breakdown_has_occurred", "secular_breakdown"),
                                    ("dynamical_instability_has_occurred", "instability_criterion")):
                if getattr(self.remote[3], attribute):
                    events.append(name)
            if not events:
                raise RuntimeError("Root return without an identified event.")
        return elapsed, "+".join(events)

    def snapshot(self):
        u, p = self.units, self.remote
        row = {"mass_MSun": float(p[0].mass.value_in(u.MSun)),
               "radius_AU": float(p[0].radius.value_in(u.au)), "hamiltonian": self.H}
        normals = []
        for suffix, i in (("in", 3), ("out", 4)):
            row["a_"+suffix+"_AU"] = float(p[i].semimajor_axis.value_in(u.au))
            row["e_"+suffix] = float(p[i].eccentricity)
            h = np.asarray(p[i].h_vec.value_in(u.MSun*u.au**2/u.yr), dtype=float)
            for axis, value in zip("xyz", h):
                row[f"h_{suffix}_{axis}"] = float(value)
            normals.append(h/np.linalg.norm(h))
        row["mutual_inclination_deg"] = float(np.rad2deg(np.arccos(np.clip(np.dot(*normals), -1, 1))))
        return row

    def stop(self):
        try:
            self.solver.stop()
        except Exception as exc:
            warnings.warn(f"Worker cleanup: {exc!r}")


### Choose the integration intervals

We use the characteristic Lidov–Kozai scale

$$t_{\rm LK}=\frac{8}{15\pi}\frac{M_{\rm tot}}{m_{\rm BD}}
\frac{P_{\rm out}^2}{P_{\rm in}}(1-e_{\rm out}^2)^{3/2},\qquad
P=2\pi\sqrt{\frac{a^3}{GM}}.$$

The base interval is at most $t_{\rm LK,min}/200$, estimated along the track with adiabatic semimajor axes. We keep every stellar knot, then try subdivisions 1, 2 and 4 with the source fixed.

References once again: [Hamers & Portegies Zwart (2016)](https://arxiv.org/abs/1511.00944) and [Antognini (2015)](https://arxiv.org/abs/1504.05957).

In [ ]:
def coupled_target_grid(window, cfg, subdivisions):
    t, mass = window["time_yr"], window["mass_MSun"]
    mp, mb = cfg["m_planet_MSun"], cfg["m_bd_MSun"]
    ain = cfg["a_in_AU"]*(mass[0]+mp)/(mass+mp)
    aout = cfg["a_out_AU"]*(mass[0]+mp+mb)/(mass+mp+mb)
    cap = float(np.min(coupled_lk_time(mass, ain, aout, cfg)))/200
    # Keep every source knot. Subdivision changes integration calls, not the driver.
    parts = [np.array([0.0])]
    for left, right in zip(t[:-1], t[1:]):
        n = max(1, int(np.ceil((right-left)/cap))) * int(subdivisions)
        parts.append(np.linspace(left, right, n+1)[1:])
    return np.concatenate(parts)


def coupled_summarize(label, h, cfg, outcome):
    s = {"run": label, "outcome": outcome, "n_states": len(h["time_yr"]),
         "end_time_yr": float(h["time_yr"][-1]), "end_age_Myr": float(h["age_Myr"][-1]),
         "final_mass_MSun": float(h["mass_MSun"][-1]), "final_a_in_AU": float(h["a_in_AU"][-1]),
         "sampled_emax": float(h["e_in"].max()), "sampled_qmin_AU": float(h["q_in_AU"].min()),
         "minimum_sampled_clearance_AU": float(h["clearance_AU"].min()),
         "max_rel_mass_driver_error": float(np.max(abs(h["mass_MSun"]/h["driver_mass_MSun"]-1))),
         "max_rel_radius_driver_error": float(np.max(abs(h["radius_AU"]/h["driver_radius_AU"]-1)))}
    for suffix in ("in", "out"):
        s[f"max_aM_{suffix}_error"] = float(np.max(abs(h[f"aM_{suffix}_ratio"]-1)))
    return s


def coupled_transfer_checks(s):
    return {"mass_driver": s["max_rel_mass_driver_error"] < COUPLED_LIMITS["mass_driver_relative"],
            "radius_driver": s["max_rel_radius_driver_error"] < COUPLED_LIMITS["radius_driver_relative"],
            "aM_in": s["max_aM_in_error"] < COUPLED_LIMITS["aM_relative"],
            "aM_out": s["max_aM_out_error"] < COUPLED_LIMITS["aM_relative"]}


### Run and compare cases

We compare whole trajectories at common times.

In [ ]:
def coupled_run_case(plan, label, window, cfg, quadrupole=True, subdivisions=1,
             rtol=1e-13, atol=1e-15, events=True, worker_options=None, backend_factory=None):
    folder = plan["folder"]/label
    folder.mkdir(exist_ok=False)
    targets = coupled_target_grid(window, cfg, subdivisions)
    times, masses, radii = (window[k] for k in ("time_yr", "mass_MSun", "radius_AU"))
    mdot, rdot = np.diff(masses)/np.diff(times), np.diff(radii)/np.diff(times)
    info = {"status": "started", "configuration": cfg, "quadrupole": quadrupole,
            "subdivisions": subdivisions, "rtol": rtol, "atol_e": atol,
            "event_detection": events, "source": plan["provenance"],
            "age_offset_Myr": window["age_offset_Myr"], "planned_outputs": len(targets)}
    coupled_save_json(folder/"run.json", info)
    rows, backend, outcome = [], None, "completed_window"
    started = last_print = perf_counter()
    factory = CoupledAmuseBackend if backend_factory is None else backend_factory
    print(f"{label}: {len(targets):,} planned outputs", flush=True)
    try:
        with coupled_working_directory(folder):
            backend = factory(cfg, masses[0], radii[0], quadrupole, rtol, atol, events, worker_options or {})
            previous = 0.0
            for j, target in enumerate(targets):
                event = ""
                if j:
                    k = min(int(np.searchsorted(times, previous, side="right")-1), len(mdot)-1)
                    backend.set_rates(mdot[k], rdot[k])
                    duration = float(target-previous)
                    elapsed, event = backend.advance(duration)
                    if not np.isfinite(elapsed) or elapsed < 0 or elapsed > duration+1e-7:
                        raise RuntimeError("Invalid returned integration duration.")
                    if not event and not np.isclose(elapsed, duration, rtol=1e-12, atol=1e-7):
                        raise RuntimeError("Worker did not reach the requested time.")
                    actual = previous+elapsed if event else float(target)
                else:
                    actual = 0.0
                row = backend.snapshot()
                row.update(time_yr=actual, age_Myr=window["age_offset_Myr"]+actual/1e6,
                           driver_mass_MSun=float(np.interp(actual, times, masses)),
                           driver_radius_AU=float(np.interp(actual, times, radii)))
                for suffix in ("in", "out"):
                    a, e = row[f"a_{suffix}_AU"], row[f"e_{suffix}"]
                    if a <= 0 or not 0 <= e < 1:
                        raise RuntimeError("Unbound or invalid orbit returned.")
                    extra = cfg["m_planet_MSun"] + (cfg["m_bd_MSun"] if suffix == "out" else 0)
                    row[f"aM_{suffix}_ratio"] = a*(row["mass_MSun"]+extra)/(cfg[f"a_{suffix}_AU"]*(masses[0]+extra))
                row["q_in_AU"] = row["a_in_AU"]*(1-row["e_in"])
                row["clearance_AU"] = row["q_in_AU"]-row["radius_AU"]-COUPLED_RPLANET_AU
                if not np.isfinite(list(row.values())).all() or row["mass_MSun"] <= 0 or row["radius_AU"] <= 0:
                    raise RuntimeError("Non-finite or non-positive physical state.")
                rows.append(row)
                previous = actual
                if event:
                    outcome = event
                    info["event"] = {"kind": event, "time_yr": actual, "age_Myr": row["age_Myr"],
                                     "clearance_AU": row["clearance_AU"], "previous_output_time_yr": rows[-2]["time_yr"] if j else actual}
                    break
                if events and row["clearance_AU"] <= 0:
                    raise RuntimeError("Surface contact without a root return; investigate event detection.")
                if perf_counter()-last_print >= 30:
                    print(f"  {j+1:,}/{len(targets):,}; elapsed {(perf_counter()-started)/60:.1f} min", flush=True)
                    last_print = perf_counter()
    except BaseException as exc:
        info.update(status="failed", exception=repr(exc), saved_states=len(rows))
        if rows:
            np.savez_compressed(folder/"incomplete_history.npz", **{k: np.array([r[k] for r in rows]) for k in rows[0]})
        coupled_save_json(folder/"run.json", info)
        raise
    finally:
        if backend is not None:
            backend.stop()
    h = {k: np.array([r[k] for r in rows]) for k in rows[0]}
    summary = coupled_summarize(label, h, cfg, outcome)
    checks = coupled_transfer_checks(summary)
    np.savez_compressed(folder/"history.npz", **h)
    coupled_save_json(folder/"summary.json", summary)
    info.update(status="completed" if all(checks.values()) else "failed_transfer_checks",
                outcome=outcome, coupled_transfer_checks=checks, wall_seconds=perf_counter()-started)
    coupled_save_json(folder/"run.json", info)
    if not all(checks.values()):
        raise RuntimeError(f"{label}: transfer checks failed: {checks}; saved in {folder}")
    print(f"  {outcome}; e_max={summary['sampled_emax']:.6f}; {info['wall_seconds']:.1f} s", flush=True)
    return h, summary


In [ ]:
def coupled_compare(left, right):
    a, b = left[0], right[0]
    common = a["time_yr"][a["time_yr"] <= b["time_yr"][-1]]
    count = len(common)
    if count < 2:
        raise RuntimeError("Insufficient common trajectory for comparison.")
    result = {"left": left[1]["run"], "right": right[1]["run"], "common_states": count}
    for key, name, relative in (("e_in", "max_abs_delta_e", False), ("q_in_AU", "max_abs_delta_q_AU", False),
                                 ("a_in_AU", "max_rel_delta_a_in", True), ("a_out_AU", "max_rel_delta_a_out", True)):
        ref = np.interp(common, b["time_yr"], b[key])
        delta = abs(a[key][:count]-ref)
        result[name] = float(np.max(delta/ref if relative else delta))
    result["same_outcome"] = left[1]["outcome"] == right[1]["outcome"]
    result["end_time_difference_yr"] = abs(left[1]["end_time_yr"]-right[1]["end_time_yr"])
    result["passed"] = bool(result["same_outcome"] and result["end_time_difference_yr"] < COUPLED_LIMITS["event_time_yr"]
                            and result["max_abs_delta_e"] < COUPLED_LIMITS["comparison_e_absolute"]
                            and result["max_abs_delta_q_AU"] < COUPLED_LIMITS["comparison_q_AU"]
                            and max(result["max_rel_delta_a_in"], result["max_rel_delta_a_out"]) < COUPLED_LIMITS["comparison_a_relative"])
    return result


def coupled_plot_results(plan, results):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
    for label, text in (("03_evolving_no_torque", "Perturbations disabled"),
                        ("04_evolving_n1", "Quadrupole, base intervals"),
                        ("07_evolving_n4_tight", "Quadrupole, finest/tight")):
        h = results[label][0]
        x = h["time_yr"]/1e6
        axes[0].plot(x, h["e_in"], label=text)
        axes[1].plot(x, h["q_in_AU"], label=text)
        axes[2].plot(x, h["aM_in_ratio"]-1, label=text)
    window = plan["window"]
    axes[1].plot(window["time_yr"]/1e6, window["radius_AU"]+COUPLED_RPLANET_AU, "k--", label=r"$R_\star+R_p$")
    axes[0].set_ylabel("Inner eccentricity")
    axes[0].legend(frameon=False)
    axes[1].set_ylabel("Pericentre / contact distance [AU]")
    axes[1].legend(frameon=False)
    axes[2].set(xlabel="Time since AGB-window start [Myr]", ylabel=r"$aM/(a_0M_0)-1$")
    fig.tight_layout()
    fig.savefig(plan["folder"]/"coupled_agb_window.png", dpi=180)
    plt.show()


### Check the fixed-mass limit first

For a nearly circular test-particle orbit at quadrupole order,

$$e_{\max}\simeq\sqrt{1-\frac53\cos^2 I_0}\simeq0.76376\quad (I_0=60^\circ).$$

This is an approximate benchmark for our small but finite planet mass. We also check angular momentum and the secular Hamiltonian in the fixed-mass runs.

Inspired by the older Lidov Kozai papers aswell as the zhang et al 2026 system (fix up these references later) we check the following cases

| Cases | Check |
| --- | --- |
| 00–01 | Fixed mass, standard and tighter tolerances |
| 02 | Fixed mass at $30^\circ$: nearly circular |
| 03 | Evolving star, torques off: constant eccentricity |
| 04–06 | Evolving star, subdivisions 1/2/4 |
| 07 | Subdivision 4, tighter solver tolerances |

All evolving cases here use the same stellar driver.

In [ ]:
def coupled_run_suite(plan, worker_options=None, backend_factory=None):
    """Run conservative checks, then a matched control and the AGB experiment."""
    folder, cfg, window = plan["folder"], plan["cfg"], plan["window"]
    if (folder/"suite_status.json").exists():
        raise FileExistsError("This batch was already started. Re-run the preparation cell for a new batch.")
    status = {"status": "started", "scope": "AGB window; fixed stellar driver; no tides or survival claim"}
    coupled_save_json(folder/"suite_status.json", status)
    env = {"python": sys.version, "executable": sys.executable, "platform": platform.platform(),
           "worker_options": worker_options or {}, "implementation": COUPLED_IMPLEMENTATION,
           "notebook": "notebooks/04_mass_loss_coupling.ipynb",
           "backend": "AMUSE socket/MPI worker" if backend_factory is None else "development test backend"}
    if "environment" in globals():
        env["setup_environment"] = environment
    for name in ("amuse-framework", "amuse-secularmultiple", "numpy"):
        try:
            env[name] = version(name)
        except PackageNotFoundError:
            env[name] = "not installed"
    coupled_save_json(folder/"environment.json", env)
    results = {}
    def run(label, source=window, configuration=cfg, **options):
        result = coupled_run_case(plan, label, source, configuration, worker_options=worker_options,
                          backend_factory=backend_factory, **options)
        results[label] = result
        coupled_save_rows(folder/"summary.csv", [r[1] for r in results.values()])
        return result
    try:
        characteristic = float(coupled_lk_time(cfg["m_star_MSun"], cfg["a_in_AU"], cfg["a_out_AU"], cfg))
        frozen = {"time_yr": np.array([0., 20*characteristic]),
                  "mass_MSun": np.full(2, cfg["m_star_MSun"]), "radius_AU": np.full(2, 1.2*COUPLED_RSUN_AU),
                  "age_offset_Myr": 0.0}
        fixed = run("00_fixed_mass", source=frozen, events=False)
        fixed_tight = run("01_fixed_mass_tight", source=frozen, rtol=1e-14, atol=1e-16, events=False)
        low = run("02_low_inclination", source=frozen, configuration={**cfg, "mutual_inclination_deg": 30.0}, events=False)
        fixed_checks = {}
        for label, (h, s) in (("fixed", fixed), ("tight", fixed_tight), ("low", low)):
            J = np.column_stack([h[f"h_in_{axis}"]+h[f"h_out_{axis}"] for axis in "xyz"])
            j_error = float(np.max(np.linalg.norm(J-J[0], axis=1))/np.linalg.norm(J[0]))
            H = h["hamiltonian"]
            h_error = float(np.max(abs(H-H[0]))/max(float(np.max(abs(H))), np.finfo(float).tiny))
            fixed_checks[label+"_J"] = j_error < COUPLED_LIMITS["fixed_J_relative"]
            fixed_checks[label+"_H"] = h_error < COUPLED_LIMITS["fixed_H_relative"]
            s.update(max_rel_total_J_error=j_error, max_scaled_H_error=h_error)
            coupled_save_json(folder/s["run"]/"summary.json", s)
        analytic = float(np.sqrt(max(0, 1-(5/3)*np.cos(np.deg2rad(cfg["mutual_inclination_deg"]))**2)))
        fixed_comparison = coupled_compare(fixed, fixed_tight)
        fixed_checks.update(emax_near_test_particle_limit=abs(fixed_tight[1]["sampled_emax"]-analytic)<COUPLED_LIMITS["benchmark_emax_absolute"],
                            low_inclination_nearly_circular=low[1]["sampled_emax"]<0.01,
                            solver_tolerance_stable=fixed_comparison["passed"])
        coupled_save_json(folder/"fixed_mass_checks.json", {"checks": fixed_checks, "analytic_emax": analytic,
                                                    "comparison": fixed_comparison})
        if not all(fixed_checks.values()):
            raise RuntimeError(f"Fixed-mass benchmark requires review: {fixed_checks}")
        isolated = run("03_evolving_no_torque", quadrupole=False)
        e_constant = all(float(np.max(abs(isolated[0]["e_"+s]-cfg["e_"+s]))) < 1e-10 for s in ("in", "out"))
        if not e_constant:
            raise RuntimeError("Eccentricity changed in the no-torque control.")
        coarse = run("04_evolving_n1")
        medium = run("05_evolving_n2", subdivisions=2)
        fine = run("06_evolving_n4", subdivisions=4)
        tight = run("07_evolving_n4_tight", subdivisions=4, rtol=1e-14, atol=1e-16)
        comparisons = [coupled_compare(coarse, medium), coupled_compare(medium, fine), coupled_compare(fine, tight)]
        coupled_save_rows(folder/"refinement_comparisons.csv", comparisons)
        valid_outcomes = all(r[1]["outcome"] in ("completed_window", "surface_contact") for r in results.values())
        passed = bool(all(c["passed"] for c in comparisons) and valid_outcomes)
        status.update(status="passed_working_tolerances" if passed else "needs_review", fixed_mass_checks=fixed_checks,
                      no_torque_eccentricity_constant=e_constant, comparisons=comparisons,
                      selected_run="07_evolving_n4_tight" if passed else None,
                      source=plan["provenance"], source_sampling_convergence_of_orbital_phase="not tested here",
                      geometric_contact_is_not_resolved_engulfment=True)
        coupled_save_rows(folder/"summary.csv", [r[1] for r in results.values()])
        coupled_save_json(folder/"suite_status.json", status)
        coupled_plot_results(plan, results)
        print(json.dumps(status, indent=2))
        print("Read:", folder/"summary.csv")
        print("Read:", folder/"refinement_comparisons.csv")
        return {"folder": folder, "status": status, "runs": results}
    except BaseException as exc:
        status.update(status="failed_or_interrupted", exception=repr(exc), completed_runs=list(results))
        coupled_save_json(folder/"suite_status.json", status)
        raise


### Run the eight-case AGB pilot

The first cell prepares the source; the second runs the suite. SHould take about 15-30 minutes.


In [ ]:
coupled_plan = coupled_prepare(
    ROOT,
    track_path=STELLAR_TRACK,
    a_in_AU=5.0,
    a_out_AU=100.0,
    inclination_deg=60.0,
)


In [ ]:
coupled_result = coupled_run_suite(coupled_plan, worker_options=WORKER_OPTIONS)


### Read the results



In [ ]:
coupled_folder = coupled_result["folder"]
print("AGB pilot:", coupled_result["status"]["status"])
with (coupled_folder / "summary.csv").open(encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        print(f"{row['run']}: {row['outcome']}; "
              f"minimum clearance = {float(row['minimum_sampled_clearance_AU']):.8f} au")
print("\nSaved files:")
for filename in ("summary.csv", "refinement_comparisons.csv", "suite_status.json"):
    print((coupled_folder / filename).relative_to(ROOT).as_posix())


## 7. Change the stellar sampling

Each pair starts at the same stellar age with the same orbital elements, but uses its own sampled mass and radius. We use the union of the two grids and tighter solver settings, keeping all source knots.

The helpers prepare, run and report either pair. Our limits stay at $10^{-3}$ in eccentricity, 0.001 au in pericentre and minimum clearance, and 10 years for end-time differences. Contact timing is checked only if both runs report contact.

In [ ]:
# Prepare the two stellar drivers on a common age interval.
def prepare_stellar_pair(plan, levels=(64, 128), stellar_dir=None):
    if stellar_dir is None:
        stellar_dir = (ROOT / plan["provenance"]["source_track"]).parent.parent
    stellar_dir = project_file(ROOT, stellar_dir)
    paths = {
        n: stellar_dir / f"refinement_{n:03d}/stellar_track.npz"
        for n in levels
    }

    windows, inputs = {}, {}
    for n, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Missing stellar track: {path}")
        with np.load(path, allow_pickle=False) as data:
            inputs[n] = [
                float(data[k])
                for k in ("initial_mass_MSun", "metallicity")
            ]
            if int(data["request_refinement"]) != n:
                raise ValueError(f"Unexpected refinement in {path}")
        windows[n] = coupled_load_agb_window(path)

    np.testing.assert_allclose(
        inputs[levels[0]], inputs[levels[1]], rtol=0, atol=1e-12,
    )

    start = max(w["age_offset_Myr"] for w in windows.values())
    times = {
        n: w["time_yr"] + (w["age_offset_Myr"] - start)*1e6
        for n, w in windows.items()
    }
    duration = min(t[-1] for t in times.values())
    if duration <= 0:
        raise ValueError("The AGB windows do not overlap.")

    cfg = {}
    for n, w in windows.items():
        t = times[n]
        cut = np.unique(
            np.r_[0.0, t[(t > 0) & (t < duration)], duration]
        )
        windows[n] = {
            "time_yr": cut,
            "age_offset_Myr": start,
            **{
                k: np.interp(cut, t, w[k])
                for k in ("mass_MSun", "radius_AU")
            },
        }
        cfg[n] = {
            **plan["cfg"],
            "m_star_MSun": float(windows[n]["mass_MSun"][0]),
        }

    # Keep every source knot and use the same integration calls.
    grid = np.unique(np.concatenate([
        coupled_target_grid(windows[n], cfg[n], 4)
        for n in levels
    ]))

    for n, w in windows.items():
        windows[n] = {
            "time_yr": grid.copy(),
            "age_offset_Myr": start,
            **{
                k: np.interp(grid, w["time_yr"], w[k])
                for k in ("mass_MSun", "radius_AU")
            },
        }
        np.testing.assert_array_equal(
            coupled_target_grid(windows[n], cfg[n], 1), grid,
        )

        q0 = cfg[n]["a_in_AU"]*(1 - cfg[n]["e_in"])
        contact_radius = windows[n]["radius_AU"][0] + COUPLED_RPLANET_AU
        if q0 <= contact_radius:
            raise ValueError("Initial orbit is already in surface contact.")

    limits = {
        "eccentricity_absolute": 1e-3,
        "pericentre_AU": 1e-3,
        "minimum_clearance_AU": 1e-3,
        "event_time_yr": 10.0,
    }

    folder = plan["folder"] / (
        f"stellar_sampling_{levels[0]:03d}_{levels[1]:03d}_"
        + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "_" + uuid.uuid4().hex[:6]
    )
    folder.mkdir(exist_ok=False)

    coupled_save_json(folder / "settings.json", {
        "source_tracks": {
            str(n): p.relative_to(ROOT).as_posix()
            for n, p in paths.items()
        },
        "start_age_Myr": start,
        "end_age_Myr": start + duration/1e6,
        "configuration": {str(n): c for n, c in cfg.items()},
        "working_tolerances": limits,
        "rtol": 1e-14,
        "atol_e": 1e-16,
        "scope": (
            "Common stellar ages; identical initial orbital elements; no tides."
        ),
    })

    print(
        f"Common stellar ages: "
        f"{start:.9f} to {start + duration/1e6:.9f} Myr"
    )
    print(f"Planned outputs per run: {len(grid):,}")
    print("Output:", folder.relative_to(ROOT))

    return dict(
        levels=levels, paths=paths, windows=windows, cfg=cfg,
        limits=limits, folder=folder, results={},
    )


In [ ]:
def run_stellar_pair(pair):
    for n in pair["levels"]:
        if n in pair["results"]:
            continue

        pair_plan = {
            "folder": pair["folder"],
            "provenance": {
                "source_track": (
                    pair["paths"][n].relative_to(ROOT).as_posix()
                ),
                "stellar_refinement": n,
                "comparison": "common stellar ages and identical integration calls",
            },
        }

        np.savez_compressed(
            pair_plan["folder"] / f"source_window_{n:03d}.npz",
            **pair["windows"][n],
        )

        pair["results"][n] = coupled_run_case(
            pair_plan,
            f"stellar_n{n:03d}",
            pair["windows"][n],
            pair["cfg"][n],
            subdivisions=1,
            rtol=1e-14,
            atol=1e-16,
            worker_options=WORKER_OPTIONS,
        )

        coupled_save_rows(
            pair_plan["folder"] / "summary.csv",
            [s for h, s in pair["results"].values()],
        )
    return pair["results"]


In [ ]:
# Compare the trajectories and the closest approaches.
def report_stellar_pair(pair):
    coarse, fine = pair["levels"]
    hc, sc = pair["results"][coarse]
    hf, sf = pair["results"][fine]

    end = min(hc["time_yr"][-1], hf["time_yr"][-1])
    t = np.unique(np.r_[
        hc["time_yr"][hc["time_yr"] <= end],
        hf["time_yr"][hf["time_yr"] <= end],
    ])
    if len(t) < 2:
        raise RuntimeError("Insufficient common trajectory to compare.")

    delta = {
        k: (
            np.interp(t, hc["time_yr"], hc[k])
            - np.interp(t, hf["time_yr"], hf[k])
        )
        for k in ("e_in", "q_in_AU", "clearance_AU", "a_in_AU")
    }

    comparison = {
        "coarse": coarse,
        "fine": fine,
        "max_abs_delta_e": float(np.max(abs(delta["e_in"]))),
        "max_abs_delta_q_AU": float(np.max(abs(delta["q_in_AU"]))),
        "max_abs_delta_clearance_AU": float(
            np.max(abs(delta["clearance_AU"]))
        ),
        "difference_minimum_clearance_AU": abs(
            sc["minimum_sampled_clearance_AU"]
            - sf["minimum_sampled_clearance_AU"]
        ),
        "max_rel_delta_a_in": float(np.max(
            abs(delta["a_in_AU"])
            / np.interp(t, hf["time_yr"], hf["a_in_AU"])
        )),
        "same_outcome": sc["outcome"] == sf["outcome"],
        "end_time_difference_yr": abs(
            sc["end_time_yr"] - sf["end_time_yr"]
        ),
    }

    limits = pair["limits"]
    both_contact = sc["outcome"] == sf["outcome"] == "surface_contact"
    end_times_match = (
        comparison["end_time_difference_yr"] < limits["event_time_yr"]
    )

    checks = {
        "eccentricity": (
            comparison["max_abs_delta_e"] < limits["eccentricity_absolute"]
        ),
        "pericentre": (
            comparison["max_abs_delta_q_AU"] < limits["pericentre_AU"]
        ),
        "minimum_clearance": (
            comparison["difference_minimum_clearance_AU"]
            < limits["minimum_clearance_AU"]
        ),
        "matching_outcome": (
            comparison["same_outcome"]
            and sc["outcome"] in ("completed_window", "surface_contact")
        ),
        "matching_end_time": end_times_match,
        "contact_time": end_times_match if both_contact else None,
    }

    closest = []
    for n, (h, summary) in pair["results"].items():
        j = int(np.argmin(h["clearance_AU"]))
        closest.append({
            "refinement": n,
            "outcome": summary["outcome"],
            "minimum_sampled_clearance_AU": float(h["clearance_AU"][j]),
            "age_at_minimum_clearance_Myr": float(h["age_Myr"][j]),
            "sampled_emax": float(np.max(h["e_in"])),
        })

    passed = all(v for v in checks.values() if v is not None)
    report = {
        "status": (
            f"passed_{coarse}_vs_{fine}_working_tolerances"
            if passed else "needs_review"
        ),
        "checks": checks,
        "comparison": comparison,
        "closest_approaches": closest,
        "scope": (
            "This AGB pilot over the common stellar-age interval; "
            "no tides or full-history survival inference."
        ),
        "note": (
            "Contact timing is checked only when both runs "
            "report surface contact."
        ),
    }

    coupled_save_rows(pair["folder"] / "comparison.csv", [comparison])
    coupled_save_rows(pair["folder"] / "closest_approaches.csv", closest)
    coupled_save_json(pair["folder"] / "validation_status.json", report)
    print(json.dumps(report, indent=2))

    fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
    for n, (h, summary) in pair["results"].items():
        axes[0].plot(
            h["time_yr"]/1e6, h["e_in"],
            label=f"Stellar refinement {n}",
        )
        axes[1].plot(h["time_yr"]/1e6, h["clearance_AU"])

    axes[0].set_ylabel("Inner eccentricity")
    axes[0].legend(frameon=False)
    axes[1].axhline(0, color="k", linestyle="--", linewidth=1)
    axes[1].set_ylabel("Surface clearance [AU]")
    axes[2].plot(t/1e6, delta["q_in_AU"])
    axes[2].set(
        xlabel="Time since common start [Myr]",
        ylabel=rf"$q_{{{coarse}}}-q_{{{fine}}}$ [AU]",
    )
    fig.tight_layout()
    fig.savefig(
        pair["folder"] / "stellar_sampling_comparison.png", dpi=180,
    )
    plt.show()
    return report


### Compare 32 and 64

This pair uses the AGB pilot's stellar batch. Should take about 15 minutes for both runs.

In [ ]:
if coupled_result["status"]["status"] != "passed_working_tolerances":
    raise RuntimeError("Complete the coupled pilot checks first.")

pair_32_64 = prepare_stellar_pair(coupled_plan, levels=(32, 64))


In [ ]:
sampling_results = run_stellar_pair(pair_32_64)


In [ ]:
report_32_64 = report_stellar_pair(pair_32_64)


## 8. Explain the pericentre difference

In the saved 32–64 result, minimum clearance passes but the pericentre history does not. Separate the difference exactly:

$$q_{32}-q_{64}=(a_{32}-a_{64})(1-e_{64})-a_{32}(e_{32}-e_{64}).$$

The first term comes from $a$, the second from $e$. We compare the measured change in $a$ with the mass-driven prediction.

In [ ]:
# Use the histories from the completed 32–64 comparison.
h32, s32 = pair_32_64["results"][32]
h64, s64 = pair_32_64["results"][64]
sampling_cfg = pair_32_64["cfg"]
sampling_folder = pair_32_64["folder"]
start_age = pair_32_64["windows"][32]["age_offset_Myr"]
end = min(h32["time_yr"][-1], h64["time_yr"][-1])
compare_times = np.unique(np.r_[
    h32["time_yr"][h32["time_yr"] <= end],
    h64["time_yr"][h64["time_yr"] <= end],
])

# Separate the pericentre difference into changes in a and e.
def on_comparison_grid(history, key):
    return np.interp(compare_times, history["time_yr"], history[key])

a32 = on_comparison_grid(h32, "a_in_AU")
a64 = on_comparison_grid(h64, "a_in_AU")
e32 = on_comparison_grid(h32, "e_in")
e64 = on_comparison_grid(h64, "e_in")
m32 = on_comparison_grid(h32, "driver_mass_MSun")
m64 = on_comparison_grid(h64, "driver_mass_MSun")

q_from_a = (a32-a64)*(1-e64)
q_from_e = -a32*(e32-e64)
delta_q = q_from_a + q_from_e
peak = int(np.argmax(abs(delta_q)))

# Predict a(t) directly from each supplied mass history.
predicted_a = {}
for n, mass in ((32, m32), (64, m64)):
    cfg = sampling_cfg[n]
    mp = cfg["m_planet_MSun"]
    predicted_a[n] = (
        cfg["a_in_AU"]*(cfg["m_star_MSun"]+mp)/(mass+mp)
    )

diagnostic = {
    "peak_stellar_age_Myr": float(start_age + compare_times[peak]/1e6),
    "delta_q_AU": float(delta_q[peak]),
    "contribution_from_a_AU": float(q_from_a[peak]),
    "contribution_from_e_AU": float(q_from_e[peak]),
    "driver_mass_32_MSun": float(m32[peak]),
    "driver_mass_64_MSun": float(m64[peak]),
    "measured_delta_a_AU": float(a32[peak]-a64[peak]),
    "mass_predicted_delta_a_AU": float(
        predicted_a[32][peak]-predicted_a[64][peak]
    ),
}

print(json.dumps(diagnostic, indent=2))
coupled_save_json(
    sampling_folder / "pericentre_diagnostic.json", diagnostic,
)

# Inspect the stellar mass and orbital discrepancy around the largest difference.
x = compare_times/1e6
half_width = 0.05  # Myr on either side of the peak.

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(x, m32, label="Stellar refinement 32")
axes[0].plot(x, m64, "--", label="Stellar refinement 64")
axes[0].set_ylabel(r"Stellar mass [$M_\odot$]")
axes[0].legend(frameon=False)

axes[1].plot(x, delta_q, label="Total pericentre difference")
axes[1].plot(x, q_from_a, "--", label="Contribution from a")
axes[1].plot(x, q_from_e, ":", label="Contribution from e")
axes[1].set(
    xlabel="Time since common start [Myr]",
    ylabel="Pericentre difference [AU]",
    xlim=(max(0, x[peak]-half_width),
          min(x[-1], x[peak]+half_width)),
)
axes[1].legend(frameon=False)

fig.tight_layout()
fig.savefig(sampling_folder / "pericentre_diagnostic.png", dpi=180)
plt.show()


### Compare the analytical expansion at 32, 64 and 128

About 99.3% of the saved peak pericentre difference comes from $a$. We compare

$$a_n(t)=a_0\frac{M_n(t_0)+m_p}{M_n(t)+m_p}.$$

We use the selected stellar batch for all three tracks, over common ages with source knots and midpoints.

In [ ]:
# Compare the saved 32, 64 and 128 stellar histories.
mass_track_dir = STELLAR_REFINEMENT_DIR
print("Stellar inputs:", mass_track_dir.relative_to(ROOT))

mass_paths = {
    n: mass_track_dir / f"refinement_{n:03d}/stellar_track.npz"
    for n in (32, 64, 128)
}

mass_windows, mass_inputs = {}, {}

for n, path in mass_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"The required stellar track is missing: {path}")

    with np.load(path, allow_pickle=False) as data:
        mass_inputs[n] = [
            float(data[k]) for k in ("initial_mass_MSun", "metallicity")
        ]

    mass_windows[n] = coupled_load_agb_window(path)

for n in (32, 128):
    np.testing.assert_allclose(
        mass_inputs[n], mass_inputs[64], rtol=0, atol=1e-12,
    )

# Evaluate both refinement pairs over exactly the same stellar ages.
mass_start_age = max(w["age_offset_Myr"] for w in mass_windows.values())
mass_times = {
    n: w["time_yr"] + (w["age_offset_Myr"]-mass_start_age)*1e6
    for n, w in mass_windows.items()
}
mass_duration = min(t[-1] for t in mass_times.values())

if mass_duration <= 0:
    raise ValueError("The three AGB windows do not overlap.")

mass_knots = np.unique(np.concatenate([
    np.array([0.0, mass_duration]),
    *[
        t[(t > 0) & (t < mass_duration)]
        for t in mass_times.values()
    ],
]))

# Include midpoints as well as source knots when sampling the nonlinear a(M) law.
mass_grid = np.unique(np.r_[
    mass_knots,
    (mass_knots[:-1]+mass_knots[1:])/2,
])

mass_curves, expansion_curves = {}, {}
a0 = coupled_plan["cfg"]["a_in_AU"]
mp = coupled_plan["cfg"]["m_planet_MSun"]

for n, w in mass_windows.items():
    mass_curves[n] = np.interp(
        mass_grid, mass_times[n], w["mass_MSun"],
    )
    expansion_curves[n] = (
        a0*(mass_curves[n][0]+mp)/(mass_curves[n]+mp)
    )

mass_comparisons = []

for coarse, fine in ((32, 64), (64, 128)):
    delta_a = expansion_curves[coarse]-expansion_curves[fine]
    peak_index = int(np.argmax(abs(delta_a)))

    mass_comparisons.append({
        "coarse": coarse,
        "fine": fine,
        "max_sampled_abs_delta_a_AU": float(np.max(abs(delta_a))),
        "max_sampled_rel_delta_a": float(
            np.max(abs(delta_a)/expansion_curves[fine])
        ),
        "max_sampled_abs_delta_mass_MSun": float(
            np.max(abs(mass_curves[coarse]-mass_curves[fine]))
        ),
        "peak_delta_a_age_Myr": float(
            mass_start_age+mass_grid[peak_index]/1e6
        ),
    })

coarse_difference, fine_difference = [
    row["max_sampled_abs_delta_a_AU"] for row in mass_comparisons
]
reduction = (
    coarse_difference/fine_difference
    if fine_difference > 0 else None
)

mass_report = {
    "status": "diagnostic_only",
    "source_tracks": {
        str(n): path.relative_to(ROOT).as_posix()
        for n, path in mass_paths.items()
    },
    "start_age_Myr": mass_start_age,
    "end_age_Myr": mass_start_age+mass_duration/1e6,
    "a0_AU": a0,
    "planet_mass_MSun": mp,
    "comparisons": mass_comparisons,
    "reduction_factor": reduction,
    "scope": (
        "Adiabatic mass-driven expansion only; "
        "eccentricity and contact not recalculated."
    ),
}

coupled_save_rows(
    sampling_folder / "mass_refinement_comparison.csv",
    mass_comparisons,
)
coupled_save_json(
    sampling_folder / "mass_refinement_diagnostic.json",
    mass_report,
)
print(json.dumps(mass_report, indent=2))

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
x = mass_grid/1e6

for n in (32, 64, 128):
    axes[0].plot(x, mass_curves[n], label=f"Stellar refinement {n}")

for coarse, fine in ((32, 64), (64, 128)):
    axes[1].plot(
        x,
        expansion_curves[coarse]-expansion_curves[fine],
        label=f"{coarse} minus {fine}",
    )

axes[0].set_ylabel(r"Stellar mass [$M_\odot$]")
axes[0].legend(frameon=False)
axes[1].set(
    xlabel="Time since common start [Myr]",
    ylabel=r"$\Delta a_{\rm adiabatic}$ [AU]",
)
axes[1].legend(frameon=False)

fig.tight_layout()
fig.savefig(
    sampling_folder / "mass_refinement_comparison.png",
    dpi=180,
)
plt.show()


## 9. Repeat with stellar refinements 64 and 128

We check the actual trajectories using the same selected stellar batch. Both runs share ages and integration calls, and we keep the original tolerances. Took 30 minutes for this.

In [ ]:
pair_64_128 = prepare_stellar_pair(
    coupled_plan, levels=(64, 128), stellar_dir=STELLAR_REFINEMENT_DIR,
)


In [ ]:
run_stellar_pair(pair_64_128);


In [ ]:
report_64_128 = report_stellar_pair(pair_64_128)



The pericentre history still needs review even though the closest approaches agree. Reminder this was AGB-only.